# Lane B pilot — Akamai upload notebook (EXP-LANEB-GDAISY-001-PILOT)
**Purpose:** run the B0–B3 pilot against *your* workshop vLLM endpoint and produce one zip to hand back.

1. Upload this file to JupyterLab (any folder **outside** the workshop checkout). 2. Run workshop `00_connect_and_verify` first (confirms the endpoint). 3. Run **all cells** here. 4. Download the zip named at the end.

- Reads `VLLM_HOST`, `MODEL_NAME`, `VLLM_API_KEY` from the environment. **The key is never written to any file or into the zip.**
- Read-only against vLLM: no Kubernetes writes, no model switching. Sequential requests only.
- Pilot = **non-confirmatory**. ECA oracle task; scored later, independently, off-server. Anticube / ΔG* = NOT_COMPUTED.
- Status: NOT EXECUTED until the zip exists.

In [ ]:
import os, sys, json, pathlib, subprocess, hashlib, platform, time, zipfile
try:
    import requests, numpy
except ImportError as e:
    raise SystemExit(f"missing dependency: {e.name} (do not pip install inside the workshop checkout; ask for help)")
HOST  = os.environ.get("VLLM_HOST", "")          # e.g. http://vllm:8000/v1
MODEL = os.environ.get("MODEL_NAME", "")
assert HOST and MODEL, "VLLM_HOST / MODEL_NAME not set in this kernel's environment"
PER_RULE = 4          # items per ECA rule (4 rules) -> 16 items; raise to 8 for the full 32-item pilot
MAX_TOKENS_PER_CALL = 512
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
WORK = pathlib.Path(f"lane_b_akamai_{STAMP}"); (WORK/"harness").mkdir(parents=True); (WORK/"raw_metrics").mkdir()
print("work dir:", WORK, "| model:", MODEL, "| endpoint origin:", HOST.split("/v1")[0])

In [ ]:
# embedded harness file: items.py
(WORK/'harness'/'items.py').write_text('"""Item generation. Matches recovered RUN001 `question()` (numpy default_rng, width 8). NO oracle here."""\nimport numpy as np\n\nRULES = (30, 90, 110, 184)\n\n\ndef make_item(rule: int, seed: int, width: int = 8) -> dict:\n    x = np.random.default_rng(seed).integers(0, 2, size=width, dtype=np.int64)\n    return {"item_id": f"ECA-{rule}-{seed}", "rule": rule, "seed": seed, "width": width, "initial_state": x.tolist()}\n\n\ndef make_items(seed_start: int, per_rule: int) -> list[dict]:\n    """per_rule items for each rule, consecutive seeds from seed_start (disjoint from calibration 41031-41034)."""\n    items, s = [], seed_start\n    for rule in RULES:\n        for _ in range(per_rule):\n            assert s not in (41031, 41032, 41033, 41034)\n            items.append(make_item(rule, s))\n            s += 1\n    return items\n')

In [ ]:
# embedded harness file: run_lane_b.py
(WORK/'harness'/'run_lane_b.py').write_text('#!/usr/bin/env python3\n"""Run Lane B arms B0-B3 against an OpenAI-compatible endpoint (Ollama /v1, vLLM, ...).\n\nThis script never computes or sees oracle outputs. It writes answers.jsonl and COMMIT.json\n(sha256 of answers.jsonl); score_lane_b.py verifies the commit, then scores independently.\nUsage: run_lane_b.py CONFIG.json OUT_DIR\n"""\nimport hashlib, json, random, re, sys, time\nfrom pathlib import Path\nimport requests\nsys.path.insert(0, str(Path(__file__).parent))\nfrom items import make_items\n\nARMS = ["B0_FIXED", "B1_GENESIS", "B2_GENESIS_DAISY", "B3_RANDOM_CONTROL"]\nGENESIS_POLICY = "GENESIS_POLICY_PROXY_V0_NOT_THE_RECOVERED_GENESIS_TRANSFORMER"\n\nSYS = "You simulate elementary cellular automata exactly. End with a final line \'ANSWER: <8 bits>\' (or \'ANSWER: ABSTAIN\')."\n\n\ndef prob(it):\n    return (f"Elementary cellular automaton, Wolfram rule {it[\'rule\']}, width 8, periodic (wrap-around) boundary. "\n            f"Neighborhood (left,center,right) as a 3-bit number L*4+C*2+R; new cell = bit (that number) of the rule number. "\n            f"Current state (index 0..7): {\'\'.join(map(str, it[\'initial_state\']))}. Give the state after ONE step.")\n\n\ndef parse(text):\n    m = re.findall(r"ANSWER:\\s*([01]{8}|ABSTAIN)", text or "")\n    return m[-1] if m else None\n\n\ndef make_chat(cfg):\n    import os\n    def chat(messages):\n        key = os.environ.get(cfg.get("api_key_env", ""), "")  # read at call time; never stored in config, answers or receipts\n        body = {"model": cfg["model"], "messages": messages, "temperature": cfg["temperature"],\n                "seed": cfg["decode_seed"], "max_tokens": cfg["max_tokens_per_call"], **cfg.get("extra_body", {})}\n        t0 = time.monotonic()\n        r = requests.post(cfg["base_url"].rstrip("/") + "/chat/completions", timeout=cfg.get("timeout_s", 300),\n                          headers={"Authorization": "Bearer " + (key or "none")}, json=body)\n        dt = time.monotonic() - t0\n        r.raise_for_status()\n        j = r.json()\n        text = j["choices"][0]["message"].get("content") or ""\n        usage = (j.get("usage") or {}).get("completion_tokens")  # None if absent; never inferred\n        return text, usage, dt\n    return chat\n\n\ndef call(chat, rec, user):\n    text, usage, dt = chat([{"role": "system", "content": SYS}, {"role": "user", "content": user}])\n    rec["calls"].append({"completion_tokens": usage, "wall_s": dt, "text_sha256": hashlib.sha256(text.encode()).hexdigest(),\n                         "text": text})\n    return text\n\n\ndef run_item(arm, it, chat, cfg):\n    rec = {"item_id": it["item_id"], "arm": arm, "calls": []}\n    p = prob(it)\n    if arm == "B0_FIXED":\n        ans = [parse(call(chat, rec, p))]\n    elif arm == "B1_GENESIS":  # 3 independent derivations (distinct prompt variants), majority vote\n        v = [p, p + " Work cell by cell, listing each neighborhood.", p + " Verify by recomputing the table of the rule first."]\n        ans = [parse(call(chat, rec, x)) for x in v]\n    else:  # B2 / B3: answer, challenge, adjudicate (identical call count and schedule)\n        a1 = parse(call(chat, rec, p))\n        if arm == "B2_GENESIS_DAISY":  # informed, oracle-free challenge\n            c = call(chat, rec, p + f" A candidate answer is {a1}. Independently find any cell that is wrong and give the corrected state.")\n        else:  # randomized uninformative challenge: seeded cell index, no information about correctness\n            rng = random.Random(f"{cfg[\'decode_seed\']}-{it[\'item_id\']}")\n            c = call(chat, rec, p + f" A candidate answer is {a1}. Please re-check cell {rng.randrange(8)} and restate the full state.")\n        a2 = parse(c)\n        adj = parse(call(chat, rec, p + f" Two candidate answers: A={a1}, B={a2}. Pick the correct one by recomputing; reply \'ANSWER: <8 bits>\'."))\n        ans = [adj]\n    valid = [a for a in ans if a]\n    final = sorted(valid, key=lambda a: (-valid.count(a), a))[0] if valid else None  # majority; deterministic tie-break\n    toks = [c["completion_tokens"] for c in rec["calls"]]\n    rec.update({"final_answer": final, "tokens_total": None if None in toks else sum(toks),\n                "wall_s_total": sum(c["wall_s"] for c in rec["calls"]), "n_calls": len(rec["calls"]),\n                "over_budget": bool(None not in toks and sum(toks) > cfg["item_token_cap"])\n                or len(rec["calls"]) > cfg["item_call_cap"]})\n    return rec\n\n\ndef main(cfg_path, out, chat=None):\n    cfg = json.loads(Path(cfg_path).read_text())\n    out = Path(out); out.mkdir(parents=True, exist_ok=True)\n    chat = chat or make_chat(cfg)\n    items = make_items(cfg["seed_start"], cfg["per_rule"])\n    (out / "items.jsonl").write_text("".join(json.dumps(i, sort_keys=True) + "\\n" for i in items))\n    with open(out / "answers.jsonl", "w") as f:\n        for arm in ARMS:\n            for it in items:\n                try:\n                    rec = run_item(arm, it, chat, cfg)\n                except Exception as e:  # failures are retained, not dropped\n                    rec = {"item_id": it["item_id"], "arm": arm, "calls": [], "final_answer": None, "failure": type(e).__name__,\n                           "tokens_total": None, "wall_s_total": 0.0, "n_calls": 0, "over_budget": False}\n                f.write(json.dumps(rec, sort_keys=True) + "\\n")\n    sha = hashlib.sha256((out / "answers.jsonl").read_bytes()).hexdigest()\n    (out / "COMMIT.json").write_text(json.dumps({"answers_sha256": sha, "config_sha256": hashlib.sha256(Path(cfg_path).read_bytes()).hexdigest(),\n                                                "genesis_policy": GENESIS_POLICY, "oracle_accessed": False}, indent=2))\n    print("answers committed", sha)\n\n\nif __name__ == "__main__":\n    main(*sys.argv[1:3])\n')

In [ ]:
# embedded harness file: score_lane_b.py
(WORK/'harness'/'score_lane_b.py').write_text('#!/usr/bin/env python3\n"""Independent scorer: verifies the answer commit, recomputes the ECA oracle on its own, writes exchange rows + analysis.\n\nUsage: score_lane_b.py CONFIG.json RUN_DIR [--source-class OBSERVED|SYNTHETIC_FIXTURE]\nOracle is re-implemented here deliberately; it is not imported from the runner.\n"""\nimport hashlib, json, sys\nfrom math import comb\nfrom pathlib import Path\n\nARMS = ["B0_FIXED", "B1_GENESIS", "B2_GENESIS_DAISY", "B3_RANDOM_CONTROL"]\n\n\ndef oracle(it):  # periodic boundary, Wolfram encoding L*4+C*2+R (matches recovered RUN001 independent_oracle)\n    b, r, n = it["initial_state"], it["rule"], len(it["initial_state"])\n    return "".join(str((r >> ((b[(i - 1) % n] << 2) | (b[i] << 1) | b[(i + 1) % n])) & 1) for i in range(n))\n\n\ndef mcnemar_exact(b, c):  # b: A right/B wrong, c: A wrong/B right; two-sided exact\n    n = b + c\n    if n == 0:\n        return 1.0\n    k = min(b, c)\n    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)\n\n\ndef holm(ps):\n    order = sorted(range(len(ps)), key=lambda i: ps[i]); adj = [0.0] * len(ps); run = 0.0\n    for rank, i in enumerate(order):\n        run = max(run, min(1.0, (len(ps) - rank) * ps[i])); adj[i] = run\n    return adj\n\n\ndef main(cfg_path, run_dir, source_class="OBSERVED"):\n    cfg = json.loads(Path(cfg_path).read_text()); run = Path(run_dir)\n    commit = json.loads((run / "COMMIT.json").read_text())\n    if hashlib.sha256((run / "answers.jsonl").read_bytes()).hexdigest() != commit["answers_sha256"]:\n        sys.exit("ANSWER COMMIT MISMATCH: refusing to score")\n    items = {i["item_id"]: i for i in map(json.loads, (run / "items.jsonl").read_text().splitlines())}\n    recs = [json.loads(l) for l in (run / "answers.jsonl").read_text().splitlines()]\n    pilot = cfg.get("mode") != "CONFIRMATORY_FROZEN"\n    split = "PILOT_NONCONFIRMATORY" if pilot else "HELDOUT_FROZEN"\n    correct = {a: {} for a in ARMS}; stats = {a: dict(ham=0.0, abst=0, parse=0, tok=0, tok_unknown=0, wall=0.0, n=0, fail=0) for a in ARMS}\n    for r in recs:\n        it = items[r["item_id"]]; truth = oracle(it); fa = r["final_answer"]; s = stats[r["arm"]]; s["n"] += 1\n        s["wall"] += r["wall_s_total"]; s["fail"] += 1 if r.get("failure") else 0\n        if r["tokens_total"] is None: s["tok_unknown"] += 1\n        else: s["tok"] += r["tokens_total"]\n        if fa == "ABSTAIN": s["abst"] += 1\n        if fa is None: s["parse"] += 1\n        ok = fa == truth and not r["over_budget"]  # parse failure, abstention, over-budget all count as incorrect\n        correct[r["arm"]][r["item_id"]] = ok\n        s["ham"] += (sum(x != y for x, y in zip(fa, truth)) / len(truth)) if fa and fa != "ABSTAIN" else 1.0\n    pairs = [("B2_GENESIS_DAISY", "B3_RANDOM_CONTROL"), ("B1_GENESIS", "B0_FIXED"), ("B2_GENESIS_DAISY", "B1_GENESIS")]\n    res = []\n    for a, b in pairs:\n        ids = sorted(correct[a]); x = sum(correct[a][i] and not correct[b][i] for i in ids); y = sum(correct[b][i] and not correct[a][i] for i in ids)\n        res.append({"A": a, "B": b, "A_only_correct": x, "B_only_correct": y, "p_exact_mcnemar": mcnemar_exact(x, y)})\n    for r_, p_ in zip(res, holm([r_["p_exact_mcnemar"] for r_ in res])): r_["p_holm"] = p_\n    (run / "analysis.json").write_text(json.dumps({"split": split, "n_items": len(items), "contrasts": res,\n        "note": "PILOT: non-confirmatory, no decision" if pilot else "confirmatory", "commit": commit}, indent=2))\n    prereg = "lane_b/PREREGISTRATION_DRAFT.md#v0.1-DRAFT-UNFROZEN"; rows = []\n    def row(arm, metric, unit, value, status="OBSERVED", note=None):\n        rows.append({"schema_version": "cell_akamai.lane_b_result.v1", "experiment_id": cfg["experiment_id"], "arm": arm,\n            "model": cfg["model"], "dataset_ref": f"ECA_NEXT_ROW_seed{cfg[\'seed_start\']}_perrule{cfg[\'per_rule\']}", "model_revision": None,\n            "metric": metric, "unit": unit, "value": value, "status": status, "evidence_ref": str(run / "answers.jsonl") + "@" + commit["answers_sha256"],\n            "evaluation_split": split, "source_class": source_class, "preregistration_ref": prereg, "commit_sha": None,\n            "fco_occurrence_id": None, "claim_ceiling": "PILOT non-confirmatory; synthetic ECA control only" if pilot else "synthetic ECA control only"})\n    for a in ARMS:\n        s = stats[a]; n = s["n"]\n        row(a, "exact_match_accuracy_primary", "fraction", sum(correct[a].values()) / n)\n        row(a, "mean_hamming_error", "fraction_of_cells", s["ham"] / n)\n        row(a, "abstention_rate", "fraction", s["abst"] / n)\n        row(a, "parse_failure_rate", "fraction", s["parse"] / n)\n        if s["tok_unknown"]:\n            row(a, "completion_tokens_total", "tokens", None, "UNKNOWN"); row(a, "request_tokens_per_s", "tokens/s", None, "UNKNOWN")\n        else:\n            row(a, "completion_tokens_total", "tokens", s["tok"]); row(a, "request_tokens_per_s", "tokens/s", s["tok"] / s["wall"] if s["wall"] else None, "OBSERVED" if s["wall"] else "UNKNOWN")\n        row(a, "wall_time_total", "s", s["wall"])\n        row(a, "anticube_operational_class", "class", None, "NOT_COMPUTED")\n        row(a, "delta_g_star", "UNDEFINED_NO_VALIDATED_FORMULA", None, "NOT_COMPUTED")\n    (run / "lane_b_results.jsonl").write_text("".join(json.dumps(x, sort_keys=True) + "\\n" for x in rows))\n    print(f"scored {len(recs)} records; split={split}; wrote {len(rows)} rows")\n\n\nif __name__ == "__main__":\n    main(sys.argv[1], sys.argv[2], *(sys.argv[3:5][1:] if len(sys.argv) > 3 else []))\n')

In [ ]:
# preflight: served model must match, save raw /metrics bytes (unmodified) before the run
base = HOST.rstrip("/"); root = base[:-3] if base.endswith("/v1") else base
hdr = {"Authorization": "Bearer " + os.environ["VLLM_API_KEY"]} if os.environ.get("VLLM_API_KEY") not in (None, "", "not-needed") else {}
served = [m["id"] for m in requests.get(base + "/models", headers=hdr, timeout=15).json()["data"]]
assert MODEL in served, f"served models {served} do not include {MODEL}"
def vget(path):
    try: r = requests.get(root + path, headers=hdr, timeout=10); r.raise_for_status(); return r.content
    except Exception as e: return None
def snap(tag):
    raw = vget("/metrics")
    if raw is None: return {"tag": tag, "status": "FAILED"}
    (WORK/"raw_metrics"/f"{tag}.prom").write_bytes(raw)
    return {"tag": tag, "status": "OBSERVED", "sha256": "sha256:" + hashlib.sha256(raw).hexdigest(), "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
ver = vget("/version"); snaps = [snap("before")]
env = {"served_models": served, "vllm_version_endpoint": ver.decode() if ver else "UNKNOWN", "python": sys.version, "platform": platform.platform(),
       "node": platform.node(), "namespace": os.environ.get("NAMESPACE", "UNKNOWN"), "model_revision": "UNKNOWN", "precision_verified": "UNKNOWN"}
env

In [ ]:
CFG = {"experiment_id": "EXP-LANEB-GDAISY-001-PILOT", "mode": "PILOT_NONCONFIRMATORY", "base_url": HOST, "model": MODEL, "temperature": 0,
       "decode_seed": 20261012, "max_tokens_per_call": MAX_TOKENS_PER_CALL, "item_token_cap": 3 * MAX_TOKENS_PER_CALL, "item_call_cap": 3,
       "seed_start": 50000, "per_rule": PER_RULE, "api_key_env": "VLLM_API_KEY",
       "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},   # Qwen3: thinking off, per the handoff protocol
       "host": "Akamai workshop " + os.environ.get("NAMESPACE", "UNKNOWN")}
(WORK/"config.json").write_text(json.dumps(CFG, indent=1))
t0 = time.time()
r = subprocess.run([sys.executable, str(WORK/"harness"/"run_lane_b.py"), str(WORK/"config.json"), str(WORK/"run")], capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:], f"| elapsed {time.time()-t0:.0f}s")
assert r.returncode == 0

In [ ]:
snaps.append(snap("after"))
s = subprocess.run([sys.executable, str(WORK/"harness"/"score_lane_b.py"), str(WORK/"config.json"), str(WORK/"run")], capture_output=True, text=True)
print(s.stdout, s.stderr[-300:])
import pandas as pd
rows = pd.read_json(WORK/"run"/"lane_b_results.jsonl", lines=True)
rows[rows.status == "OBSERVED"].pivot(index="metric", columns="arm", values="value")

In [ ]:
print(json.dumps(json.loads((WORK/"run"/"analysis.json").read_text())["contrasts"], indent=1))  # PILOT: no decision

In [ ]:
# package: manifest of sha256 for every file; the API key is never included
(WORK/"environment.json").write_text(json.dumps({**env, "metrics_snapshots": snaps}, indent=1))
files = sorted(p for p in WORK.rglob("*") if p.is_file() and p.name != "MANIFEST.json")
(WORK/"MANIFEST.json").write_text(json.dumps({"files": {str(p.relative_to(WORK)): "sha256:" + hashlib.sha256(p.read_bytes()).hexdigest() for p in files},
    "claim_state": "PILOT_NONCONFIRMATORY_EXECUTED_ON_AKAMAI_ENDPOINT", "anticube": "NOT_COMPUTED", "delta_g_star": "NOT_COMPUTED",
    "mmr": "NOT_COMMITTED", "signed": False, "independent_reexecution": "NOT_TESTED"}, indent=1))
ZIP = f"lane_b_akamai_pilot_{STAMP}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(WORK.rglob("*")):
        if p.is_file(): z.write(p, p.relative_to(WORK.parent))
print("DOWNLOAD THIS FILE from the JupyterLab file browser:", pathlib.Path(ZIP).resolve(), os.path.getsize(ZIP), "bytes")